# Scratch Analysis — Preguntas ad-hoc sobre el edge estructural

Notebook temporal, NO parte de la suite oficial. Reusa la misma base de datos y las mismas funciones de construcción de dataframes que `custom_analysis.ipynb` / `custom_analysis_llm_report.md` (cuenta `flight_account_001_xauusd`, solo lectura).

Estructura: una celda de setup (imports, conexión, carga de tablas, helpers `_build_*`) y luego una celda por pregunta, agregada incrementalmente.

In [1]:
import sys
import os
import re
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from IPython.display import display

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))  # para poder importar core/*

pd.set_option('display.float_format', lambda x: f'{x:.4f}')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 0)

from core.edge_analysis import analisis_edge, analizar_correlaciones, profit_factor_by_edge_bins, get_market_outcome

In [2]:
# Conexion de solo lectura a la DB de la cuenta 001 (xauusd)
DB_URL = "sqlite:///../.data/flight_account_001_xauusd.db"
engine = create_engine(DB_URL, connect_args={'timeout': 15})

with engine.connect() as conn:
    unified_df = pd.read_sql(text("SELECT * FROM unified_department;"), conn, parse_dates=['created_at', 'updated_at'])
    layer_df = pd.read_sql(text("SELECT * FROM analysis_layer;"), conn)
    efficiency_df = pd.read_sql(text("SELECT * FROM efficiency_audit;"), conn, parse_dates=['created_at', 'updated_at', 'resolution_time'])
    tactical_df = pd.read_sql(text("SELECT * FROM tactical_audit;"), conn)

# tactical_audit paso de 1:1 a 1:many con unified_department (ago-2026, ver CLAUDE.md).
# Esta DB real puede no estar migrada todavia -- detectar en runtime en vez de asumir.
TACTICAL_HAS_TRADE_ID = 'trade_id' in tactical_df.columns
if TACTICAL_HAS_TRADE_ID:
    # Union por trade_id (no por id, que ahora es el id propio de la ejecucion), quedandose
    # con la ejecucion mas reciente por analisis para no inflar el N con 2+ ejecuciones.
    tactical_latest_df = (
        tactical_df.sort_values('created_at')
        .drop_duplicates('trade_id', keep='last')
        .drop(columns=['id'])
        .rename(columns={'trade_id': 'id'})
    )
else:
    # Pre-migracion: id todavia es el id compartido 1:1 legacy con unified_department.
    tactical_latest_df = tactical_df.copy()

print(f"unified_department: {unified_df.shape[0]} filas")
print(f"analysis_layer: {layer_df.shape[0]} filas")
print(f"efficiency_audit: {efficiency_df.shape[0]} filas")
print(f"tactical_audit: {tactical_df.shape[0]} filas")

unified_department: 68 filas
analysis_layer: 340 filas
efficiency_audit: 68 filas
tactical_audit: 68 filas


## Helpers (idénticos a `custom_analysis.ipynb` / `custom_analysis_llm_report.md`)

In [3]:
def _build_audit_df(unified_df: pd.DataFrame, efficiency_df: pd.DataFrame) -> pd.DataFrame:
    """efficiency_audit + market_bias/probabilidades de unified_department,
    restringida a setups ya resueltos (specific_bias_compliance in {Valid, Invalid})."""
    audit_df = efficiency_df.merge(
        unified_df[['id', 'market_bias', 'calc_edge', 'long_prob', 'short_prob', 'no_trade_prob']],
        on='id', how='left',
    )
    audit_df = audit_df[audit_df['specific_bias_compliance'].isin(['Valid', 'Invalid'])].copy()
    audit_df['is_valid'] = (audit_df['specific_bias_compliance'] == 'Valid').astype(int)
    audit_df['theoretical_dir'] = np.where(audit_df['calc_edge'] >= 0, 'Long', 'Short')
    audit_df['primary_prob'] = np.where(audit_df['theoretical_dir'] == 'Long', audit_df['long_prob'], audit_df['short_prob'])
    audit_df['bias_group'] = np.where(audit_df['market_bias'] == 'Choppy / Neutral', 'Choppy', 'Direccional (Bull/Bear)')
    return audit_df


STRUCTURAL_TARGET_RR = 2.2  # os.getenv("STRUCTURAL_TARGET_RR", "2.2") en el .env real


def _build_struct_df(unified_df: pd.DataFrame, efficiency_df: pd.DataFrame) -> pd.DataFrame:
    """Todos los analisis (no solo los resueltos), con structural_entry_price,
    direcciones, y MAE/MFE crudos y normalizados a R."""
    struct_df = unified_df.merge(efficiency_df[['id', 'structural_mae', 'structural_mfe']], on='id', how='left')

    struct_df['structural_entry_price'] = np.where(
        struct_df['edge_validation_price'].notna() & struct_df['structural_invalidation'].notna(),
        (struct_df['edge_validation_price'] + STRUCTURAL_TARGET_RR * struct_df['structural_invalidation']) / (1 + STRUCTURAL_TARGET_RR),
        np.nan,
    )
    struct_df['direction_from_bias'] = np.select(
        [struct_df['market_bias'] == 'Bullish', struct_df['market_bias'] == 'Bearish'],
        ['Long', 'Short'], default=None,
    )
    struct_df['theoretical_dir'] = np.where(struct_df['calc_edge'] >= 0, 'Long', 'Short')
    struct_df['primary_prob'] = np.where(struct_df['theoretical_dir'] == 'Long', struct_df['long_prob'], struct_df['short_prob'])

    struct_df['mae_past_entry'] = np.where(
        struct_df['theoretical_dir'] == 'Long',
        struct_df['structural_entry_price'] - struct_df['structural_mae'],
        struct_df['structural_mae'] - struct_df['structural_entry_price'],
    )
    struct_df['mfe_past_entry'] = np.where(
        struct_df['theoretical_dir'] == 'Long',
        struct_df['structural_mfe'] - struct_df['structural_entry_price'],
        struct_df['structural_entry_price'] - struct_df['structural_mfe'],
    )
    struct_df['dist_mark_entry'] = struct_df['mark_price'] - struct_df['structural_entry_price']
    struct_df['dist_mark_entry_abs'] = struct_df['dist_mark_entry'].abs()

    struct_df['one_r'] = (struct_df['structural_entry_price'] - struct_df['structural_invalidation']).abs()
    struct_df['mae_R'] = struct_df['mae_past_entry'] / struct_df['one_r']
    struct_df['mfe_R'] = struct_df['mfe_past_entry'] / struct_df['one_r']
    return struct_df


def _build_px_df(unified_df: pd.DataFrame, efficiency_df: pd.DataFrame, layer_df: pd.DataFrame) -> pd.DataFrame:
    """analysis_layer pivoteado (P0..P4) + market_bias/calc_edge/specific_bias_compliance,
    con Market_Outcome, consensus_count y final_sign."""
    score_pivot = layer_df.pivot(index='trade_id', columns='layer_name', values='score')
    score_pivot = score_pivot.rename(columns={f'P{i}': f'p{i}_score' for i in range(5)})
    score_pivot = score_pivot.reset_index().rename(columns={'trade_id': 'id'})

    px_df = unified_df[['id', 'market_bias', 'calc_edge']].merge(
        efficiency_df[['id', 'specific_bias_compliance']], on='id', how='left'
    ).merge(score_pivot, on='id', how='left')

    px_df['Market_Outcome'] = px_df.apply(get_market_outcome, axis=1)
    px_df['final_sign'] = np.sign(px_df['calc_edge'])
    score_cols = ['p0_score', 'p1_score', 'p2_score', 'p3_score', 'p4_score']

    def _consensus(row):
        return sum(1 for c in score_cols if row[c] != 0 and np.sign(row[c]) == row['final_sign'])

    px_df['consensus_count'] = px_df.apply(_consensus, axis=1)
    px_df['is_valid'] = (px_df['specific_bias_compliance'] == 'Valid').astype(int)
    return px_df


TF_PATTERN = re.compile(r'(\d+[A-Za-z]*)\s*-\s*B([LR])', re.IGNORECASE)
TFS = ['1DT', '12H', '4H', '1H']


def _normalize_tf(tf: str):
    tf = tf.upper()
    if tf == '1DT':
        return '1DT'
    if tf in ('12HT', '12H', '12T'):
        return '12H'
    if tf in ('4HT', '4H', '4T'):
        return '4H'
    if tf in ('1HT', '1H', '1T'):
        return '1H'
    return None


def _build_tf_df(unified_df: pd.DataFrame, efficiency_df: pd.DataFrame, layer_df: pd.DataFrame):
    """Parsea la thesis de P0 ('{temporalidad}-B{L|R}') a una columna tf_<TF> por
    cada una de las 4 temporalidades (1DT/12H/4H/1H). Se une con market_bias/
    calc_edge/specific_bias_compliance y se calcula Market_Outcome/htf_majority."""
    layer_p0 = layer_df[layer_df['layer_name'] == 'P0'][['trade_id', 'score', 'thesis']].copy()
    records = []
    for _, r in layer_p0.iterrows():
        found = {}
        for tf_raw, direction in TF_PATTERN.findall(r['thesis']):
            tf = _normalize_tf(tf_raw)
            if tf and tf not in found:
                found[tf] = 1 if direction.upper() == 'L' else -1
        rec = {'id': r['trade_id'], 'p0_score': r['score']}
        for tf in TFS:
            rec[f'tf_{tf}'] = found.get(tf, np.nan)
        records.append(rec)
    tf_df = pd.DataFrame(records)

    tf_full = tf_df.merge(unified_df[['id', 'market_bias', 'calc_edge']], on='id', how='left')
    tf_full = tf_full.merge(efficiency_df[['id', 'specific_bias_compliance']], on='id', how='left')
    tf_full = tf_full[tf_full['specific_bias_compliance'].notna()].copy()
    tf_full['Market_Outcome'] = tf_full.apply(get_market_outcome, axis=1)

    def _majority_htf(row):
        vals = [v for v in [row['tf_1DT'], row['tf_12H'], row['tf_4H']] if pd.notna(v)]
        if not vals:
            return np.nan
        return 1 if sum(vals) > 0 else (-1 if sum(vals) < 0 else 0)

    tf_full['htf_majority'] = tf_full.apply(_majority_htf, axis=1)
    return tf_df, tf_full


def _flatten_agg_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = ['_'.join(c).strip('_') for c in df.columns]
    return df.reset_index()

In [4]:
audit_df = _build_audit_df(unified_df, efficiency_df)
struct_df = _build_struct_df(unified_df, efficiency_df)
px_df = _build_px_df(unified_df, efficiency_df, layer_df)
tf_df, tf_full = _build_tf_df(unified_df, efficiency_df, layer_df)

print(f"audit_df: {audit_df.shape[0]} filas (resueltos)")
print(f"struct_df: {struct_df.shape[0]} filas (todos)")
print(f"px_df: {px_df.shape[0]} filas")
print(f"tf_full: {tf_full.shape[0]} filas (resueltos)")

audit_df: 68 filas (resueltos)
struct_df: 68 filas (todos)
px_df: 68 filas
tf_full: 68 filas (resueltos)


---
## Preguntas ad-hoc

Cada pregunta del usuario se agrega como una celda markdown + una o mas celdas de codigo debajo, en orden.

### P1 — Slippage real: `tactical_audit.entry_price` vs. `structural_entry_price`\n\nConfirmacion directa (no por proxy) del hallazgo #7: los setups que terminan Invalid se ejecutan ~2x mas seguido que los Valid (66.7% vs 29.5%, Fisher p=0.0069, evidencia indirecta via `reached_entry`). Aqui se cruza el precio de entrada REAL registrado en `tactical_audit.entry_price` contra el `structural_entry_price` teorico, para los 27 trades con `order_filled=True`.

In [5]:
slip_df = struct_df.merge(
    tactical_latest_df[['id', 'order_filled', 'entry_price', 'r_multiple']],
    on='id', how='inner',
)
slip_df = slip_df[slip_df['order_filled'] == True].copy()
slip_df['entry_price'] = slip_df['entry_price'].astype(float)
slip_df = slip_df[slip_df['entry_price'].notna()].copy()

# Slippage: precio de entrada real vs. teorico, con signo segun theoretical_dir
# (misma convencion que mae_past_entry/mfe_past_entry en _build_struct_df:
# positivo = el fill fue MEJOR que el teorico, negativo = peor / mas tarde)
slip_df['slippage_pts'] = np.where(
    slip_df['theoretical_dir'] == 'Long',
    slip_df['structural_entry_price'] - slip_df['entry_price'],
    slip_df['entry_price'] - slip_df['structural_entry_price'],
)
slip_df['slippage_R'] = slip_df['slippage_pts'] / slip_df['one_r']

# Cruce con el resultado estructural (specific_bias_compliance) para comparar
# el fill de los setups que terminaron Valid vs. Invalid
slip_df = slip_df.merge(audit_df[['id', 'specific_bias_compliance']], on='id', how='left')

print(f"slip_df: {len(slip_df)} filas (ejecutados con entry_price no nulo, de {tactical_df['order_filled'].sum()} order_filled=True totales)")
display(slip_df[['id', 'market_bias', 'specific_bias_compliance', 'theoretical_dir', 'structural_entry_price', 'entry_price', 'slippage_pts', 'slippage_R', 'r_multiple']])

slip_df: 28 filas (ejecutados con entry_price no nulo, de 28 order_filled=True totales)


,id,market_bias,specific_bias_compliance,theoretical_dir,structural_entry_price,entry_price,slippage_pts,slippage_R,r_multiple
0,2834cbeb-b65f-4cad-97ca-1807e3193b23,Choppy / Neutral,Valid,Long,4540.0000,4547.5700,-7.5700,-0.3785,2.0718
1,0a68d499-0175-4048-acc4-8412edbb51c4,Choppy / Neutral,Valid,Long,4475.3125,4481.7000,-6.3875,-0.3145,2.7863
2,22acd14d-4e98-48e5-b6e5-41628f214d3d,Bearish,Valid,Short,4526.5625,4525.6700,-0.8925,-0.0664,0.0523
3,67b3f403-64d5-458c-832d-c00f77b1c09a,Choppy / Neutral,Valid,Short,4506.3750,4501.1500,-5.2250,-0.2533,2.4034
4,9fb17739-cb19-4771-836a-d3b6b28591bb,Choppy / Neutral,Invalid,Long,4485.0000,4488.2700,-3.2700,-0.1308,-1.0000
5,7b6e9c10-50a6-4503-9044-04223b7a658b,Bullish,Invalid,Long,4342.9375,4338.8000,4.1375,0.1595,-1.0000
6,42a48654-51d8-467a-81e1-1e5a230380df,Bearish,Valid,Short,4182.5000,4156.6700,-25.8300,-0.6888,-1.0000
7,695b2b4b-fbc0-461e-a108-01576bad312e,Bearish,Invalid,Short,4075.0000,4094.5000,19.5000,0.7800,-0.0349
8,4b17b903-407d-4a5e-b238-1491ab64679b,Choppy / Neutral,Valid,Long,4335.6250,4334.0400,1.5850,0.1014,-0.8162
9,9602d430-4569-4682-b22c-9fb689d6fd4d,Bearish,Invalid,Short,4315.1069,4312.7400,-2.3669,-0.1527,-1.0000


In [6]:
slip_by_compliance = slip_df.groupby('specific_bias_compliance')[['slippage_pts', 'slippage_R']].agg(['count', 'mean', 'median']).round(3)
slip_by_compliance.columns = ['_'.join(c).strip('_') for c in slip_by_compliance.columns]
slip_by_compliance = slip_by_compliance.reset_index()
display(slip_by_compliance)

print()
print("Referencia (evidencia indirecta ya confirmada, tasa de ejecucion por resultado):")
print("  Valid   -> ejecutado 29.5% de las veces")
print("  Invalid -> ejecutado 66.7% de las veces")
print("  (Fisher exact p=0.0069)")
print()
print(f"N total en este cruce: {len(slip_df)} (de 27 order_filled=True; "
      f"{tactical_df['order_filled'].sum() - len(slip_df)} sin entry_price registrado)")

,specific_bias_compliance,slippage_pts_count,slippage_pts_mean,slippage_pts_median,slippage_R_count,slippage_R_mean,slippage_R_median
0,Invalid,15,-0.3910,-0.7570,15,-0.0120,-0.0180
1,Valid,13,-7.6450,-6.3870,13,-0.1970,-0.2770



Referencia (evidencia indirecta ya confirmada, tasa de ejecucion por resultado):
  Valid   -> ejecutado 29.5% de las veces
  Invalid -> ejecutado 66.7% de las veces
  (Fisher exact p=0.0069)

N total en este cruce: 28 (de 27 order_filled=True; 0 sin entry_price registrado)


### P2 -- Cruce order_filled x reached_entry (miss mecanico vs. posible salto discrecional)

Objetivo: separar, entre los `order_filled=False`, cuales fue porque el precio nunca llego al `structural_entry_price` calculado (miss mecanico) vs. cuales el precio si llego pero no se tomo la orden (revisar caso por caso). Corre sobre TODO el historico actual de la cuenta (no solo los resueltos).

Nota de precision sobre el prompt original: `order_filled` vive en `tactical_audit`, no en `efficiency_audit`/`unified_department` -- se agrega el join correspondiente. `bias_a` viene de `efficiency_audit`.

Advertencia metodologica (igual que en el spec de backtesting, Seccion 1.4): `structural_mae` es el extremo GLOBAL de toda la ventana de evaluacion, sin orden temporal confiable frente a `structural_invalidation`. `reached_entry` es un proxy para priorizar revision manual, no un veredicto definitivo.

In [7]:
checkable = struct_df[
    struct_df['theoretical_dir'].notna()
    & struct_df['structural_mae'].notna()
    & struct_df['structural_entry_price'].notna()
].copy()

checkable['reached_entry'] = np.where(
    checkable['theoretical_dir'] == 'Short',
    checkable['structural_mae'] >= checkable['structural_entry_price'],
    checkable['structural_mae'] <= checkable['structural_entry_price'],
)

checkable = checkable.merge(tactical_latest_df[['id', 'order_filled']], on='id', how='left')
checkable = checkable.merge(efficiency_df[['id', 'bias_a']], on='id', how='left')

print(f"checkable: {len(checkable)} filas (de {len(struct_df)} analisis totales; "
      f"{len(struct_df) - len(checkable)} excluidos por falta de theoretical_dir/structural_mae/structural_entry_price)")
print(f"order_filled nulo en checkable: {checkable['order_filled'].isna().sum()}")

tabla = pd.crosstab(checkable['reached_entry'], checkable['order_filled'], margins=True, margins_name='Total')
display(tabla)

checkable: 59 filas (de 68 analisis totales; 9 excluidos por falta de theoretical_dir/structural_mae/structural_entry_price)
order_filled nulo en checkable: 0


order_filled,0,1,Total
reached_entry,,,
False,22,2,24
True,9,26,35
Total,31,28,59


In [8]:
cols = ['id', 'market_bias', 'bias_a', 'created_at', 'structural_entry_price', 'structural_mae']

susp_df = checkable[
    (checkable['reached_entry'] == True) & (checkable['order_filled'] == False)
][cols].sort_values('created_at').reset_index(drop=True)

print(f"Grupo a revisar caso por caso (reached_entry=True, order_filled=False): {len(susp_df)} casos")
display(susp_df)

Grupo a revisar caso por caso (reached_entry=True, order_filled=False): 9 casos


,id,market_bias,bias_a,created_at,structural_entry_price,structural_mae
0,35be0fc7-b3ba-42d7-9571-63ef2e8100af,Choppy / Neutral,Validated Range Expansion,2026-05-25 06:51:45.509490,4553.3125,4509.0000
1,70ea32f5-6b5b-4cd5-bc9d-a44e58415058,Bearish,Validated Range Expansion,2026-05-27 06:30:00.000000,4463.2500,4467.7000
2,a4d31d37-7fcb-4296-ab57-53787c960e19,Choppy / Neutral,BOS,2026-05-29 06:30:00.000000,4515.4375,4515.4000
3,2d75c2d5-d4d1-4438-84e7-945f779e31ce,Choppy / Neutral,Trend Reversal,2026-06-12 07:38:15.178288,4199.4375,4236.5800
4,d55f09b2-f542-4d0e-81ab-334ada101e7d,Bullish,Trend Reversal,2026-06-26 05:26:02.574408,4045.0000,4039.0000
5,7ea316f3-8171-4422-934d-295ec2e2dc4b,Bullish,Validated Range Expansion,2026-07-02 06:35:42.851145,4114.2500,4102.6300
6,f24b9653-ad3b-4eb0-9a78-badb9a64a09f,Bearish,BOS,2026-07-09 05:30:00.000000,4102.5000,4137.0000
7,db7de070-0d59-4ea5-84a1-0c49739e585d,Bearish,Trend Reversal,2026-07-22 05:47:19.417505,4123.3125,4168.6900
8,d6318133-4c46-4384-a949-71382a39bead,Bearish,Choppy-Bearish Range Rotation,2026-08-03 05:40:53.654814,4069.2500,4071.0000


In [9]:
rare_df = checkable[
    (checkable['reached_entry'] == False) & (checkable['order_filled'] == True)
][cols].sort_values('created_at').reset_index(drop=True)

print(f"Grupo raro (reached_entry=False, order_filled=True) -- posible entrada a mercado o error de dato: {len(rare_df)} casos")
display(rare_df)

Grupo raro (reached_entry=False, order_filled=True) -- posible entrada a mercado o error de dato: 2 casos


,id,market_bias,bias_a,created_at,structural_entry_price,structural_mae
0,b2c485fd-6e04-45ae-aa3b-733567f07cd4,Bearish,CHOCH,2026-06-18 06:50:18.276594,4283.1250,4278.0000
1,8096ad6b-0807-483e-b7de-135a4fa90a37,Choppy / Neutral,Trend Reversal,2026-07-03 06:27:42.971802,4181.2500,4180.2700


### P3 -- Recalibracion del stop (structural_invalidation)

Objetivo: determinar si el stop real coincide con el teorico (Parte A), estimar el retroceso adverso tipico en trades ganadores (Parte B, exploratorio -- sin price_bars), y simular anchos de stop alternativos (Parte C).

**Estado de price_bars: NO disponible.** Confirmado en esta misma sesion (investigacion de codigo previa a este cruce): no existe ninguna tabla, ingesta ni cliente de feed OHLC/tick en el repo -- `TRADINGVIEW_DOWNLOADS_DIR` solo se usa para screenshots PNG, no exporta precio. El usuario decidio diferir la fuente de `price_bars` (ver plan de sesion). Consecuencia directa para este prompt: Parte B se entrega solo en su version exploratoria (proxy `mae_R`, sin orden temporal), y **Parte C queda bloqueada** -- se entrega solo A y B exploratoria, como indica el propio prompt para este escenario.

Campo de stop real: `tactical_audit.stop_loss` -- SI existe (confirmado, `tools/database.py`), poblado por el wizard (`cli/main.py`, prompt "Stop Loss"). No hay que reportarlo como faltante.

In [10]:
# Parte A: stop real (tactical_audit.stop_loss) vs. stop teorico (structural_invalidation)
stop_df = struct_df.merge(
    tactical_latest_df[['id', 'order_filled', 'entry_price', 'stop_loss', 'r_multiple']],
    on='id', how='inner',
)
stop_df = stop_df[stop_df['order_filled'] == True].copy()
stop_df['stop_loss'] = stop_df['stop_loss'].astype(float)
stop_df = stop_df[stop_df['stop_loss'].notna() & stop_df['structural_invalidation'].notna()].copy()

# Convencion: positivo = stop real MAS ANCHO (mas lejos del entry) que el teorico (mas seguro).
# negativo = stop real MAS ANGOSTO (mas cerca) que el teorico (mas riesgo de barrido prematuro).
stop_df['stop_slippage_pts'] = np.where(
    stop_df['theoretical_dir'] == 'Long',
    stop_df['structural_invalidation'] - stop_df['stop_loss'],
    stop_df['stop_loss'] - stop_df['structural_invalidation'],
)
stop_df['stop_slippage_R'] = stop_df['stop_slippage_pts'] / stop_df['one_r']

stop_df = stop_df.merge(audit_df[['id', 'specific_bias_compliance']], on='id', how='left')

print(f"stop_df: {len(stop_df)} filas (ejecutados con stop_loss y structural_invalidation no nulos, de {int(tactical_df['order_filled'].sum())} order_filled=True totales)")
display(stop_df[['id', 'market_bias', 'specific_bias_compliance', 'theoretical_dir', 'structural_invalidation', 'stop_loss', 'stop_slippage_pts', 'stop_slippage_R', 'r_multiple']])

stop_df: 28 filas (ejecutados con stop_loss y structural_invalidation no nulos, de 28 order_filled=True totales)


,id,market_bias,specific_bias_compliance,theoretical_dir,structural_invalidation,stop_loss,stop_slippage_pts,stop_slippage_R,r_multiple
0,2834cbeb-b65f-4cad-97ca-1807e3193b23,Choppy / Neutral,Valid,Long,4520.0000,4531.4200,-11.4200,-0.5710,2.0718
1,0a68d499-0175-4048-acc4-8412edbb51c4,Choppy / Neutral,Valid,Long,4455.0000,4467.6600,-12.6600,-0.6233,2.7863
2,22acd14d-4e98-48e5-b6e5-41628f214d3d,Bearish,Valid,Short,4540.0000,4538.4900,-1.5100,-0.1124,0.0523
3,67b3f403-64d5-458c-832d-c00f77b1c09a,Choppy / Neutral,Valid,Short,4527.0000,4515.6500,-11.3500,-0.5503,2.4034
4,9fb17739-cb19-4771-836a-d3b6b28591bb,Choppy / Neutral,Invalid,Long,4460.0000,4471.2000,-11.2000,-0.4480,-1.0000
5,7b6e9c10-50a6-4503-9044-04223b7a658b,Bullish,Invalid,Long,4317.0000,4313.4200,3.5800,0.1380,-1.0000
6,42a48654-51d8-467a-81e1-1e5a230380df,Bearish,Valid,Short,4220.0000,4173.9000,-46.1000,-1.2293,-1.0000
7,695b2b4b-fbc0-461e-a108-01576bad312e,Bearish,Invalid,Short,4100.0000,4125.1800,25.1800,1.0072,-0.0349
8,4b17b903-407d-4a5e-b238-1491ab64679b,Choppy / Neutral,Valid,Long,4320.0000,4305.6400,14.3600,0.9190,-0.8162
9,9602d430-4569-4682-b22c-9fb689d6fd4d,Bearish,Invalid,Short,4330.6100,4330.6100,0.0000,0.0000,-1.0000


In [11]:
stop_by_compliance = stop_df.groupby('specific_bias_compliance')[['stop_slippage_pts', 'stop_slippage_R']].agg(['count', 'mean', 'median', 'std']).round(3)
stop_by_compliance.columns = ['_'.join(c).strip('_') for c in stop_by_compliance.columns]
stop_by_compliance = stop_by_compliance.reset_index()
display(stop_by_compliance)

overall = stop_df[['stop_slippage_pts', 'stop_slippage_R']].agg(['count', 'mean', 'median', 'std']).round(3)
print("\nOverall:")
display(overall)

,specific_bias_compliance,stop_slippage_pts_count,stop_slippage_pts_mean,stop_slippage_pts_median,stop_slippage_pts_std,stop_slippage_R_count,stop_slippage_R_mean,stop_slippage_R_median,stop_slippage_R_std
0,Invalid,15,-2.8730,-1.1000,12.4480,15,-0.0630,-0.0440,0.4050
1,Valid,13,-16.9670,-11.4200,21.9340,13,-0.3990,-0.5500,0.6300



Overall:


,stop_slippage_pts,stop_slippage_R
count,28.0000,28.0000
mean,-9.4160,-0.2190
median,-5.6550,-0.2020
std,18.5850,0.5390


In [12]:
# Parte B (EXPLORATORIO, sin price_bars): mae_R en trades ejecutados y ganadores (r_multiple > 0).
# mae_R ya esta normalizado por one_r (teorico) en struct_df.
# ADVERTENCIA: mae_R es el extremo GLOBAL de toda la ventana de evaluacion, SIN orden temporal
# confiable frente al avance a favor -- no dice si el retroceso ocurrio antes o despues de que
# el trade empezara a moverse a favor. Proxy exploratorio, no definitivo (spec Seccion 1.4).

winners_df = struct_df.merge(
    tactical_latest_df[['id', 'order_filled', 'r_multiple']], on='id', how='inner',
)
winners_df = winners_df.merge(audit_df[['id', 'specific_bias_compliance']], on='id', how='left')
winners_df = winners_df[
    (winners_df['order_filled'] == True) & (winners_df['r_multiple'] > 0) & winners_df['mae_R'].notna()
].copy()

print(f"Trades ejecutados y ganadores (order_filled=True, r_multiple>0) con mae_R disponible: {len(winners_df)} "
      f"(de {int(((tactical_df['order_filled']==True) & (tactical_df['r_multiple']>0)).sum())} ganadores realizados totales)")

mae_R_winners_summary = winners_df['mae_R'].agg(['count', 'mean', 'median', 'std', 'min', 'max']).round(3)
display(mae_R_winners_summary)

display(winners_df[['id', 'market_bias', 'specific_bias_compliance', 'r_multiple', 'mae_R']].sort_values('mae_R', ascending=False))

Trades ejecutados y ganadores (order_filled=True, r_multiple>0) con mae_R disponible: 12 (de 12 ganadores realizados totales)


count    12.0000
mean      0.7870
median    0.4590
std       0.9280
min      -0.1090
max       3.1000
Name: mae_R, dtype: float64

,id,market_bias,specific_bias_compliance,r_multiple,mae_R
58,7b9ea3f2-810b-47d1-ac70-bb717e9302ca,Bearish,Invalid,1.3656,3.1000
66,fdb60a42-83f6-41f1-abf1-4f005b3db6a8,Bearish,Invalid,0.9448,2.0435
63,9b0f4eef-327b-4823-9dcc-2a31cad87f07,Bullish,Invalid,0.5410,1.0356
50,82c1c17c-56d6-4158-90b3-332a7212bc4e,Bearish,Valid,2.1350,0.8625
35,d6300e0d-1c35-49bf-ab33-8424b6ab4fcb,Bearish,Valid,0.6072,0.8080
5,22acd14d-4e98-48e5-b6e5-41628f214d3d,Bearish,Valid,0.0523,0.4791
9,67b3f403-64d5-458c-832d-c00f77b1c09a,Choppy / Neutral,Valid,2.4034,0.4381
2,0a68d499-0175-4048-acc4-8412edbb51c4,Choppy / Neutral,Valid,2.7863,0.3570
0,2834cbeb-b65f-4cad-97ca-1807e3193b23,Choppy / Neutral,Valid,2.0718,0.3060
28,9fb9e581-95fc-4539-9d2f-407eab042bbc,Bearish,Valid,1.6862,0.0756


### Parte C -- BLOQUEADA

No se ejecuta. Razon: aunque el paso 1-3 (recalcular tamano de posicion y r_multiple con un stop mas ancho) parece hacerse sin price_bars usando entry_price/closing_price reales, esto solo es valido para trades que YA ganaron y cuyo retroceso maximo real (durante toda su vida) no hubiera tocado tampoco el stop ancho -- dato que en si mismo requeriria el mismo `mae_R` sin orden temporal confiable de la Parte B para decidir, reintroduciendo el problema que el propio prompt pide evitar. Para trades que perdieron con el stop actual, es imposible saber que hubiera pasado despues sin price_bars (exactamente el punto 4 del prompt). Siguiendo la instruccion explicita del prompt para este escenario ("si no esta disponible... marcar C como bloqueado"), se deja bloqueada por completo en vez de entregar una version parcial que reintroduce el mismo riesgo metodologico que el resto de este trabajo ha estado evitando.